# Week 6, Day 2: Finding eigenvalues: the characteristic polynomial, trace and determinant

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w006/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Eigenvalues from trace and determinant

Write `eig2(A)` that returns the two eigenvalues of a 2 by 2 matrix using only the trace and determinant and the formula above. Use `np.lib.scimath.sqrt`, which returns a complex number for a negative argument instead of `nan`. Return them as a tuple (largest real part first).

In [ ]:
import numpy as np
def eig2(A):
    tr = ...
    det = ...
    root = np.lib.scimath.sqrt(...)
    return ...

print(eig2(np.array([[2.0, 1.0], [1.0, 2.0]])), eig2(np.array([[0.0, -1.0], [1.0, 0.0]])))

In [ ]:
import numpy as np
def key(z):
    return (round(z.real, 8), round(z.imag, 8))

def test_real():
    "matches NumPy for real eigenvalues"
    for A in [np.array([[2.0, 1.0], [1.0, 2.0]]), np.array([[1.0, 2.0], [3.0, -4.0]])]:
        got = sorted(eig2(A), key=key); want = sorted(np.linalg.eigvals(A).astype(complex), key=key)
        assert np.allclose(got, want)

def test_complex():
    "matches NumPy when the eigenvalues are complex"
    A = np.array([[0.0, -1.0], [1.0, 0.0]])
    got = sorted(eig2(A), key=key)
    assert np.allclose(got, [-1j, 1j])
    B = np.array([[-0.2, -1.5], [1.5, -0.2]])
    assert np.allclose(sorted(eig2(B), key=key), sorted(np.linalg.eigvals(B), key=key))

test_real()
test_complex()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(3)
M = rng.normal(size=(4, 4))
lam = np.linalg.eigvals(M)
print("trace:", np.trace(M).round(4), "  sum of eigenvalues:", lam.sum().round(4))
print("det:  ", np.linalg.det(M).round(4), "  product of eigenvalues:", np.prod(lam).round(4))
T = np.triu(M)
print("upper triangular matrix: eigenvalues", np.sort(np.linalg.eigvals(T).real).round(3), " diagonal", np.sort(np.diag(T)).round(3))

In [ ]:
theta = np.radians(30)
for r in [0.9, 1.0]:
    R = r * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    lam = np.linalg.eigvals(R)
    print(f"r = {r}: eigenvalues {lam.round(3)}, |λ| = {np.abs(lam).round(3)}, angle = {np.degrees(np.angle(lam)).round(1)} degrees")

### Exercise: Classify a circuit from trace and determinant

Write `classify(J)` using only the trace τ and determinant Δ: a saddle if Δ is negative; otherwise stable if τ is negative and unstable if τ is positive; and a spiral if the eigenvalues are complex (τ²/4 less than Δ), a node if they are real. Return one of 'stable node', 'stable spiral', 'unstable node', 'unstable spiral', 'saddle'. Then write `J_ei(wEE, wEI, wIE, wII)` that builds the matrix above.

In [ ]:
def classify(J):
    tr, det = np.trace(J), np.linalg.det(J)
    ...

def J_ei(wEE, wEI, wIE, wII):
    return ...

print(classify(J_ei(1.5, 2.0, 2.0, 0.5)), eig2(J_ei(1.5, 2.0, 2.0, 0.5)))

In [ ]:
import numpy as np
def test_cases():
    "labels the five basic behaviours"
    assert classify(np.array([[-1.0, 0.0], [0.0, -2.0]])) == "stable node"
    assert classify(np.array([[-0.2, -1.0], [1.0, -0.2]])) == "stable spiral"
    assert classify(np.array([[0.2, -1.0], [1.0, 0.2]])) == "unstable spiral"
    assert classify(np.array([[1.0, 0.0], [0.0, 2.0]])) == "unstable node"
    assert classify(np.array([[1.0, 0.0], [0.0, -1.0]])) == "saddle"

def test_ei():
    "the default E-I circuit oscillates as it settles, and strong excitation destabilises it"
    assert classify(J_ei(1.5, 2.0, 2.0, 0.5)) == "stable spiral"
    assert classify(J_ei(4.0, 0.5, 0.5, 0.0)) in ("unstable node", "saddle")

test_cases()
test_ei()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
wEEs = np.linspace(0, 4, 81)
lams = np.array([eig2(J_ei(w, 2.0, 2.0, 0.5)) for w in wEEs])
labels = [classify(J_ei(w, 2.0, 2.0, 0.5)) for w in wEEs]
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(wEEs, lams[:, 0].real, label="real part"); ax[0].plot(wEEs, np.abs(lams[:, 0].imag), label="|imaginary part|")
ax[0].axhline(0, color="0.5", lw=0.8); ax[0].set_xlabel("recurrent excitation w_EE"); ax[0].legend(frameon=False)
ax[1].scatter(lams[:, 0].real, lams[:, 0].imag, c=wEEs, s=10); ax[1].scatter(lams[:, 1].real, lams[:, 1].imag, c=wEEs, s=10)
ax[1].axvline(0, color="0.5", lw=0.8); ax[1].set_xlabel("Re λ"); ax[1].set_ylabel("Im λ")
plt.tight_layout(); plt.show()
changes = [(round(wEEs[i], 2), labels[i]) for i in range(len(wEEs)) if i == 0 or labels[i] != labels[i - 1]]
print(changes)